# 05 - CrewAI

Fifth in the **Agent Frameworks** series. Same use case as
[01 - AIMU.ipynb](01%20-%20AIMU.ipynb); only the framework changes. See [README.md](../README.md).

## What CrewAI brings

[CrewAI](https://docs.crewai.com/) organizes work as a **crew of role-playing agents** collaborating
on tasks. This is the multi-agent contrast in the series: instead of one agent doing everything, a
**researcher** agent finds and curates papers and a **writer** agent synthesizes from the curated
corpus. A **critic** agent then reviews, and we loop on its feedback.

CrewAI has a native **Knowledge** feature (ChromaDB-backed) for grounding agents in documents, but
it is configured at crew/agent creation, which makes *dynamically* adding only relevance-confirmed
papers mid-run awkward (see notes). So the store here is a ChromaDB collection wired through tools —
the same curated, gated store as the other notebooks.

> **Script version:** [`scripts/05_crewai.py`](../scripts/05_crewai.py) runs this same workflow from the command line.

## 01 - Setup

```bash
ollama pull qwen3.6:27b
ollama pull nomic-embed-text
```

In [ ]:
import os

import dotenv

import chromadb
from chromadb.utils import embedding_functions
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

from genscai import research, paths

dotenv.load_dotenv(paths.root / ".env")

OLLAMA_URL = "http://localhost:11434"
MODEL = os.environ.get("GENSCAI_AGENT_MODEL", "qwen3.6:27b")
MAX_ROUNDS = 2

RESEARCH_QUESTION = (
    "What interventions and control strategies have recent preprints proposed or evaluated for "
    "dengue outbreaks, and what evidence do they report?"
)

llm = LLM(model=f"ollama/{MODEL}", base_url=OLLAMA_URL)

## 02 - The shared research tools

Same `genscai.research` search/fetch functions as every notebook in the series.

In [ ]:
for h in research.search_medrxiv("dengue vaccination", max_results=3):
    print(h["date"], "-", h["title"][:80])

## 03 - The document store and crew tools

A ChromaDB collection (local Ollama embeddings) is the curated store. The researcher agent reads each
abstract and calls `save_relevant_paper` **only** for matches.

In [ ]:
ef = embedding_functions.OllamaEmbeddingFunction(url=OLLAMA_URL, model_name="nomic-embed-text")
chroma = chromadb.PersistentClient(path=str(paths.output / "agent_frameworks" / "crewai_store"))
collection = chroma.get_or_create_collection("papers", embedding_function=ef)

_seen: dict[str, dict] = {}


@tool("search_preprints")
def search_preprints(query: str) -> str:
    """Search medRxiv and bioRxiv preprints for a topic. Always include the key term (e.g. 'dengue')."""
    results = research.search_medrxiv(query, max_results=5) + research.search_biorxiv(query, max_results=3)
    if not results:
        return "No results found."
    blocks = []
    for article in results:
        _seen[article["doi"]] = article
        blocks.append(
            f"DOI: {article['doi']}\nTitle: {article['title']}\nDate: {article['date']}\n"
            f"Abstract: {(article['abstract'] or '')[:600]}"
        )
    return "\n\n".join(blocks)


@tool("save_relevant_paper")
def save_relevant_paper(doi: str) -> str:
    """Save a paper confirmed relevant to the research question, identified by its DOI."""
    article = _seen.get(doi)
    if not article:
        return f"Unknown DOI {doi}. Search for it first."
    collection.upsert(
        ids=[doi],
        documents=[f"{article['title']}\n\n{article['abstract']}"],
        metadatas=[{"title": article["title"], "date": article["date"] or "", "url": article["url"] or ""}],
    )
    return f"Saved: {article['title']}"


@tool("read_saved_papers")
def read_saved_papers() -> str:
    """Read every paper saved in the local document store, to ground a synthesis."""
    data = collection.get()
    if not data["ids"]:
        return "No papers saved yet."
    blocks = []
    for doi, meta, doc in zip(data["ids"], data["metadatas"], data["documents"]):
        blocks.append(f"# {meta['title']}\nDOI: {doi} | Date: {meta['date']}\nURL: {meta['url']}\n\n{doc}")
    return "\n\n---\n\n".join(blocks)

## 04 - The crew: researcher + writer

Two role-playing agents. The researcher searches and curates; the writer synthesizes from the
curated store. A sequential `Crew` runs the research task, then the writing task.

In [ ]:
researcher = Agent(
    role="Preprint researcher",
    goal="Find and curate preprints relevant to the research question.",
    backstory="You search medRxiv and bioRxiv, judge each abstract's relevance, and save only the matches.",
    tools=[search_preprints, save_relevant_paper],
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Evidence synthesizer",
    goal="Write a concise, cited synthesis grounded only in the curated corpus.",
    backstory="You read the saved papers and write an evidence summary, citing each by title and DOI.",
    tools=[read_saved_papers],
    llm=llm,
    verbose=True,
)


def build_crew(writer_instruction):
    research_task = Task(
        description=(
            f"Research question: {RESEARCH_QUESTION}\n"
            "Search preprints (always include 'dengue' in queries). For each candidate, if its abstract "
            "is plausibly relevant, save it with save_relevant_paper. Save several relevant papers."
        ),
        expected_output="A short list of the papers you saved.",
        agent=researcher,
    )
    write_task = Task(
        description=writer_instruction,
        expected_output="A concise synthesis citing each paper by title and DOI.",
        agent=writer,
        context=[research_task],
    )
    return Crew(agents=[researcher, writer], tasks=[research_task, write_task], process=Process.sequential)


crew = build_crew(
    "Call read_saved_papers and write a synthesis that answers the research question using only those "
    "papers, citing each by title and DOI."
)
result = await crew.kickoff_async()
synthesis = result.raw
print("\n=== SYNTHESIS ===\n")
print(synthesis)

## 05 - The self-correcting loop

A critic agent reviews the synthesis. If it is not `PASS`, the writer revises (reading the same
curated store). We bound the loop at `MAX_ROUNDS`.

In [ ]:
critic = Agent(
    role="Review editor",
    goal="Judge whether a synthesis answers the question with cited evidence.",
    backstory="You reply PASS or REVISE with specific gaps.",
    llm=llm,
    verbose=False,
)


async def critique(synthesis):
    task = Task(
        description=(
            "Reply with exactly PASS if this synthesis answers the question, cites specific papers "
            f"(title + DOI), and reports evidence. Otherwise reply REVISE: <specific gap>.\n\n"
            f"Question: {RESEARCH_QUESTION}\n\nSynthesis:\n{synthesis}"
        ),
        expected_output="PASS or REVISE: <gap>",
        agent=critic,
    )
    result = await Crew(agents=[critic], tasks=[task], process=Process.sequential).kickoff_async()
    return result.raw


for round_num in range(MAX_ROUNDS):
    verdict = await critique(synthesis)
    print(f"--- critic (round {round_num + 1}): {verdict[:120]}")
    if "PASS" in verdict:
        break
    revise_crew = build_crew(
        f"Revise the synthesis using this feedback: {verdict}. Call read_saved_papers and answer the "
        f"question ({RESEARCH_QUESTION}) using only the saved papers, citing each by title and DOI."
    )
    synthesis = (await revise_crew.kickoff_async()).raw

print("\n=== FINAL SYNTHESIS ===\n")
print(synthesis)

## 06 - Inspect the curated corpus

In [ ]:
data = collection.get()
print(f"{len(data['ids'])} papers saved:")
for meta in data["metadatas"]:
    print(" -", meta["title"][:80])

## 07 - CrewAI notes

- **Multi-agent**: the distinguishing feature here — discrete `Agent`s with roles/goals collaborate on
  `Task`s in a `Crew`. The researcher/writer split mirrors how a team would divide the work.
- **Document store**: we used a ChromaDB collection via tools. CrewAI also has a native **Knowledge**
  feature (`StringKnowledgeSource`/`TextFileKnowledgeSource`, ChromaDB-backed) you attach at agent
  creation:
  ```python
  from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
  ks = StringKnowledgeSource(content=corpus_text)
  writer = Agent(..., knowledge_sources=[ks],
                 embedder={"provider": "ollama", "config": {"model": "nomic-embed-text"}})
  ```
  Because Knowledge is fixed at creation, it does not fit the *gated, mid-run* curation this use case
  needs, so we collect with tools instead.
- **The feedback loop**: hand-written around `crew.kickoff()` (CrewAI runs tasks once per kickoff).
- **Local models**: `LLM(model="ollama/<model>", base_url=...)` (CrewAI uses LiteLLM under the hood).